In [1]:
%pip -q install groq

In [2]:
import json
from math import ceil

from google.colab import userdata
from groq import Groq, APIConnectionError, APIStatusError, RateLimitError

# Read the API key you saved in Colab Secrets
client = Groq(api_key=userdata.get("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-120b"

# Sample prices for your campus print shop
catalogue = {
    "black and white printing": {"price_inr": 2, "unit": "per printed side"},
    "colour printing": {"price_inr": 10, "unit": "per printed side"},
    "spiral binding": {"price_inr": 30, "unit": "per copy"}
}

# Start each customer with an empty order
def new_order():
    return {
        "name": "",
        "bw_pages": 0,
        "colour_pages": 0,
        "copies": 1,
        "binding": False,
        "double_sided": False
    }

order = new_order()
history = []

print("Campus Print Shop catalogue is ready.")

Campus Print Shop catalogue is ready.


In [3]:
def calculate_bill(current):
    """Return the total price in rupees and the number of paper sheets."""
    bw_rate = catalogue["black and white printing"]["price_inr"]
    colour_rate = catalogue["colour printing"]["price_inr"]
    binding_rate = catalogue["spiral binding"]["price_inr"]

    pages_per_copy = current["bw_pages"] + current["colour_pages"]
    price_per_copy = (
        current["bw_pages"] * bw_rate
        + current["colour_pages"] * colour_rate
    )

    if current["binding"]:
        price_per_copy += binding_rate

    sheets_per_copy = pages_per_copy
    if current["double_sided"]:
        sheets_per_copy = ceil(pages_per_copy / 2)

    return price_per_copy * current["copies"], sheets_per_copy * current["copies"]


SYSTEM = """
You are a friendly assistant for a campus print shop.

Read the customer's message, recent conversation, current order, and catalogue.
Return a JSON object with all four fields:
- updates: only the order fields that should change.
- action: one of "menu", "quote", "summary", "pickup", or "reply".
- reply: a short answer or clarifying question when the action is "reply".
- preview: true for a hypothetical change and false for an actual order change.

Order fields:
- name: customer's name
- bw_pages and colour_pages: number of each type of page per copy
- copies: number of copies
- binding and double_sided: true or false

Rules:
1. Remember the customer's name and keep earlier order details unless changed.
2. If one existing page changes from black and white to colour, update both
   page counts so the total stays the same.
3. Use "menu" for general price questions and "quote" for a specific estimate.
   Python calculates the price using the catalogue.
4. Use "summary" when the customer asks you to repeat their order.
5. Use "pickup" for collection times. Only the shop owner can confirm a time.
6. If a detail is unclear, use "reply" and ask one short question.
7. For "What if I add binding?", preview the change. For "Add binding",
   apply the change to the order.
"""

In [4]:
sample = new_order()
sample.update(bw_pages=19, colour_pages=1, copies=2, binding=True)
price, sheets = calculate_bill(sample)
print(f"Sample bill: ₹{price} for {sheets} sheets")

Sample bill: ₹156 for 40 sheets


In [5]:
def apply_updates(current, updates):
    """Return a copy of the order with the requested changes."""
    if not isinstance(updates, dict):
        raise ValueError("Order changes must be a dictionary.")

    updated = current.copy()

    for field, value in updates.items():
        if field not in current:
            raise ValueError(f"Unknown order field: {field}")

        if type(value) is not type(current[field]):
            raise ValueError(f"Invalid value for {field}.")

        if field in ("bw_pages", "colour_pages") and value < 0:
            raise ValueError("Page counts must be zero or greater.")

        if field == "copies" and value < 1:
            raise ValueError("At least one copy is required.")

        updated[field] = value

    return updated


# A customer starts with 20 black-and-white pages.
test_order = new_order()
test_order["bw_pages"] = 20

# Preview changing one of those pages to colour.
preview = apply_updates(test_order, {"bw_pages": 19, "colour_pages": 1})
price, sheets = calculate_bill(preview)

print(f"Updated estimate: ₹{price} for {sheets} sheets")
print(f"Original order: {test_order['bw_pages']} black-and-white pages")

Updated estimate: ₹48 for 20 sheets
Original order: 20 black-and-white pages


In [6]:
def order_summary(current):
    """Return a readable summary of the customer's print order."""
    if current["bw_pages"] + current["colour_pages"] == 0:
        return "How many pages do you need, and should they be black and white or colour?"

    price, sheets = calculate_bill(current)
    name = current["name"] or "Customer"
    binding = "Spiral binding" if current["binding"] else "No binding"
    printing = "Double-sided" if current["double_sided"] else "Single-sided"

    return (
        f"Order summary for {name}\n"
        f"Black-and-white pages per copy: {current['bw_pages']}\n"
        f"Colour pages per copy: {current['colour_pages']}\n"
        f"Number of copies: {current['copies']}\n"
        f"Printing: {printing}\n"
        f"Binding: {binding}\n"
        f"Paper needed: {sheets} sheets\n"
        f"Total: ₹{price}"
    )


# Test with the sample order from Step 5.
print(order_summary(apply_updates(sample, {"name": "Poojitha"})))

Order summary for Poojitha
Black-and-white pages per copy: 19
Colour pages per copy: 1
Number of copies: 2
Printing: Single-sided
Binding: Spiral binding
Paper needed: 40 sheets
Total: ₹156


In [7]:
def chat(message):
    context = {"catalogue": catalogue, "current_order": order}
    messages = [
        {"role": "system", "content": SYSTEM + "\nShop data:\n" + json.dumps(context)},
        *history[-8:],
        {"role": "user", "content": message}
    ]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0,
            response_format={"type": "json_object"}
        )

        result = json.loads(response.choices[0].message.content or "{}")
        proposed = apply_updates(order, result["updates"])
        action = result["action"]
        preview = result["preview"]

        if type(preview) is not bool:
            raise ValueError("Invalid preview setting.")

        if action == "menu":
            reply = "\n".join(
                f"{service.capitalize()}: ₹{item['price_inr']} {item['unit']}"
                for service, item in catalogue.items()
            )
        elif action in ("quote", "summary"):
            reply = order_summary(proposed)
            if preview:
                reply = "Preview only\n" + reply
        elif action == "pickup":
            reply = "Please check with the shop owner to confirm a pickup time."
        elif action == "reply" and isinstance(result.get("reply"), str):
            reply = result["reply"]
        else:
            raise ValueError("Unknown reply format.")

        if not preview:
            order.update(proposed)

        history.extend([
            {"role": "user", "content": message},
            {"role": "assistant", "content": reply}
        ])
        return reply

    except RateLimitError:
        return "The service is busy. Please wait a minute and try again."
    except APIConnectionError:
        return "I couldn't reach Groq. Check your internet connection and try again."
    except APIStatusError as error:
        return f"Groq returned error {error.status_code}: {error.message}"
    except (ValueError, KeyError, TypeError):
        return "I couldn't understand that request. Could you rephrase it?"


# Send your first live customer message.
print(chat("Hi, I'm Poojitha. What services do you offer?"))

Black and white printing: ₹2 per printed side
Colour printing: ₹10 per printed side
Spiral binding: ₹30 per copy


In [10]:
print("Campus Print Shop is open! Type 'exit' to finish or 'reset' to start a new order.")

while True:
    message = input("You: ").strip()

    if message.lower() in ("exit", "bye", "quit"):
        print("Bot: Thank you for visiting Campus Print Shop!")
        break

    if message.lower() == "reset":
        order = new_order()
        history.clear()
        print("Bot: Your new order is ready.")
        continue

    if message:
        print("Bot:", chat(message))

Campus Print Shop is open! Type 'exit' to finish or 'reset' to start a new order.
You: Hi, I’m Poojitha. What services do you offer?
Bot: Black and white printing: ₹2 per printed side
Colour printing: ₹10 per printed side
Spiral binding: ₹30 per copy
You: 	How much is colour printing per side?
Bot: Black and white printing: ₹2 per printed side
Colour printing: ₹10 per printed side
Spiral binding: ₹30 per copy
You: 	I need one copy of 20 black-and-white pages. How much is it?
Bot: Order summary for Poojitha
Black-and-white pages per copy: 20
Colour pages per copy: 0
Number of copies: 1
Printing: Single-sided
Binding: Spiral binding
Paper needed: 20 sheets
Total: ₹70
You: What if I print it double-sided?
Bot: Preview only
Order summary for Poojitha
Black-and-white pages per copy: 20
Colour pages per copy: 0
Number of copies: 1
Printing: Double-sided
Binding: Spiral binding
Paper needed: 10 sheets
Total: ₹70
You: I’m Poojitha. I need one copy of 20 black-and-white pages, single-sided, wit